# Step 6 on Kaggle: Whisper large-v3 + Audar on the pipeline's segments

Input: the folder your pipeline produced (`segments.csv` + the `segments/` WAV files), uploaded as a Kaggle dataset.
Output: the same `segments.csv` with the ASR columns added (`transcript_whisper`, `second_model`, `transcript_second`, `cer`, `n_diff_words`, `status`, `final_transcript`), saved in `/kaggle/working/asr_out/`.

**Settings (right panel):** Internet ON, Accelerator GPU T4, and your segments dataset added under *Input*. Do NOT install NeMo here.

In [37]:
!pip install -q qwen-asr librosa soundfile jiwer pandas

In [38]:
# ===== CONFIG (edit only this cell) =====
SEGMENTS_CSV  = None     # None = auto-find segments.csv under /kaggle/input.
                         # If you attached several datasets, set it explicitly, e.g.
                         # "/kaggle/input/delhab-segments/delhab/segments.csv"
LIMIT         = None     # e.g. 20 to test on the first 20 segments; None = all

WHISPER_ID    = "openai/whisper-large-v3"
WHISPER_BATCH = 4        # lower to 2 or 1 on CUDA out-of-memory
WHISPER_BEAMS = 5

AUDAR_ID      = "audarai/Audar-ASR-V1-Turbo"
AUDAR_LANG    = "Arabic"
AUDAR_BATCH   = 8        # lower to 4 on CUDA out-of-memory

OUT_DIR       = "/kaggle/working/asr_out"
SR            = 16000

## 1) Find the segments and load the audio

In [40]:
import glob, os, numpy as np, pandas as pd, soundfile as sf, librosa
from pathlib import Path

if SEGMENTS_CSV is None:
    found = glob.glob("/kaggle/input/**/segments.csv", recursive=True)
    if not found:
        raise FileNotFoundError("No segments.csv under /kaggle/input - add your dataset in the Input panel.")
    if len(found) > 1:
        raise RuntimeError("Several segments.csv found, set SEGMENTS_CSV explicitly:\n" + "\n".join(found))
    SEGMENTS_CSV = found[0]

df = pd.read_csv(SEGMENTS_CSV)
if LIMIT:
    df = df.head(LIMIT).copy()
print("segments.csv :", SEGMENTS_CSV)
print("segments     :", len(df))
print("columns      :", list(df.columns))

# --- normalise column names ---
if "wav_file" in df.columns and "audio_file" not in df.columns:
    df["audio_file"] = df["wav_file"]
if "snr_db" in df.columns and "snr" not in df.columns:
    df["snr"] = df["snr_db"]
if "clean_noisy" in df.columns and "quality" not in df.columns:
    df["quality"] = df["clean_noisy"]
if "source_region_label" in df.columns and "label" not in df.columns:
    df["label"] = df["source_region_label"]

path_col = "audio_file" if "audio_file" in df.columns else "wav_file"

# ---------- robust path resolution ----------
csv_path = Path(SEGMENTS_CSV)
# Search upward from the csv location for the actual audio files
candidates = []
for p in df[path_col].head(5):          # test a few entries
    p = Path(str(p))
    # try relative to the csv folder and its parents
    for parent in [csv_path.parent] + list(csv_path.parents):
        trial = parent / p
        if trial.exists():
            candidates.append(parent)
            break
        # also try just the filename under a "segments" folder
        trial2 = parent / "segments" / p.name
        if trial2.exists():
            candidates.append(parent / "segments")
            break

if not candidates:
    raise FileNotFoundError(
        "Could not locate any of the wav files.\n"
        f"First entry in CSV: {df[path_col].iloc[0]}\n"
        f"CSV location: {SEGMENTS_CSV}\n"
        "Make sure the segments/ folder (with the .wav files) is uploaded together with segments.csv."
    )

BASE = candidates[0]
print("Audio base folder :", BASE)

paths = []
for f in df[path_col]:
    p = Path(str(f))
    # try the full relative path first, then just the filename
    full = BASE / p
    if full.exists():
        paths.append(full)
    else:
        paths.append(BASE / p.name)

missing = [str(p) for p in paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} wav files missing, first: {missing[0]}\n"
        "Upload the segments/ folder together with segments.csv."
    )

def load(p):
    x, sr = sf.read(str(p), dtype="float32", always_2d=False)
    if x.ndim > 1:
        x = x.mean(axis=1)
    if sr != SR:
        x = librosa.resample(x, orig_sr=sr, target_sr=SR)
    return x

audio = [load(p) for p in paths]
durs = np.array([len(a) / SR for a in audio])
print(f"audio loaded : {durs.sum()/60:.1f} min total | segment length min {durs.min():.1f}s, max {durs.max():.1f}s")
assert durs.max() <= 30.5, "A segment is longer than 30 s - the models take at most 30 s per input."
rms = np.array([float(np.sqrt((a ** 2).mean())) for a in audio])

os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = f"{OUT_DIR}/segments.csv"

segments.csv : /kaggle/input/datasets/ahmedhshousha/mydataset123/segments.csv
segments     : 386
columns      : ['start', 'end', 'label', 'snr', 'quality', 'model_A_transcript', 'model_B_transcript', 'final_transcript', 'status', 'review_status', 'automatic_status', 'segment_id', 'audio_file', 'changed_by', 'changed_at', 'change_log']
Audio base folder : /kaggle/input/datasets/ahmedhshousha/mydataset123/segments
audio loaded : 66.0 min total | segment length min 3.3s, max 15.1s


## 2) Whisper large-v3 (saves a checkpoint when done)

In [41]:
import gc, torch
from tqdm.auto import tqdm
from transformers import AutoProcessor, WhisperForConditionalGeneration

dt = torch.float16
w_proc = AutoProcessor.from_pretrained(WHISPER_ID)
w_model = WhisperForConditionalGeneration.from_pretrained(WHISPER_ID, torch_dtype=dt).to("cuda").eval()
print("Loaded", WHISPER_ID, "on", next(w_model.parameters()).device)

whisper_txt = []
for i in tqdm(range(0, len(audio), WHISPER_BATCH)):
    batch = audio[i:i + WHISPER_BATCH]
    feats = w_proc(batch, sampling_rate=SR, return_tensors="pt").input_features.to("cuda", dtype=dt)
    with torch.inference_mode():
        ids = w_model.generate(feats, language="ar", task="transcribe",
                               num_beams=WHISPER_BEAMS, max_new_tokens=300)
    texts = w_proc.batch_decode(ids, skip_special_tokens=True)
    for k, t in enumerate(texts):
        whisper_txt.append("" if rms[i + k] < 1e-3 else t.strip())     # silent -> blank (avoids hallucination)

df = df.copy()
df["model_A_transcript"] = whisper_txt
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("Checkpoint saved:", OUT_CSV)

del w_model, w_proc
gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory in use: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Loaded openai/whisper-large-v3 on cuda:0


  0%|          | 0/97 [00:00<?, ?it/s]

Checkpoint saved: /kaggle/working/asr_out/segments.csv
GPU memory in use: 0.0 GB


## 3) Audar-ASR-V1-Turbo

In [42]:
from qwen_asr import Qwen3ASRModel

major, _ = torch.cuda.get_device_capability(0)
adt = torch.bfloat16 if major >= 8 else torch.float16        # T4 has no bf16
a_model = Qwen3ASRModel.from_pretrained(AUDAR_ID, dtype=adt, device_map="cuda:0",
                                        max_inference_batch_size=AUDAR_BATCH, max_new_tokens=256)
a_model.model.to(adt)
print("Loaded", AUDAR_ID, "| dtype", adt)

audar_txt = []
for i in tqdm(range(0, len(audio), AUDAR_BATCH)):
    batch = [(a, SR) for a in audio[i:i + AUDAR_BATCH]]
    out = a_model.transcribe(audio=batch, language=[AUDAR_LANG] * len(batch))
    audar_txt.extend(r.text.strip() for r in out)

df["second_model"] = "audar"
df["model_B_transcript"] = [("" if r < 1e-3 else t) for t, r in zip(audar_txt, rms)]
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("Checkpoint saved:", OUT_CSV)

del a_model
gc.collect(); torch.cuda.empty_cache()

Loaded audarai/Audar-ASR-V1-Turbo | dtype torch.float16


  0%|          | 0/49 [00:00<?, ?it/s]

Checkpoint saved: /kaggle/working/asr_out/segments.csv


## 4) Compare and set the status
Same rule as `asr.py`: `accepted` if CER <= 0.02, `needs_review` if at most 3 words differ, otherwise `difficult`.

In [43]:
import re, jiwer

_DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
_ALEF = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا"})
def normalize(t):
    t = "" if t is None or (isinstance(t, float) and t != t) else str(t)
    t = _DIAC.sub("", t).replace("ـ", "").translate(_ALEF)
    t = t.replace("ة", "ه").replace("ى", "ي")
    t = re.sub(r"[^\w\s]", " ", t).replace("_", " ")
    return " ".join(t.split())

def decide(a, b):
    a, b = normalize(a), normalize(b)
    if not a or not b:
        return "difficult"
    w = jiwer.process_words(a, b)
    n_diff = w.substitutions + w.deletions + w.insertions
    cer = jiwer.cer(a, b)
    if cer <= 0.02:
        return "accepted"
    return "human_choice" if n_diff <= 3 else "difficult"

auto = [decide(a, b) for a, b in zip(df["model_A_transcript"], df["model_B_transcript"])]

df["automatic_status"] = auto
df["status"] = auto
df["review_status"] = ["accepted" if s == "accepted" else "pending" for s in auto]
df["final_transcript"] = [
    a if s == "accepted" else ""
    for a, s in zip(df["model_A_transcript"], auto)
]

# empty human-edit fields (this is a fresh automatic run)
df["changed_by"] = ""
df["changed_at"] = ""
df["change_log"] = "[]"

# Exact column order of the annotation app (identical to P4rayXcD5_g_review.csv)
APP_COLS = [
    "start", "end", "label", "snr", "quality",
    "model_A_transcript", "model_B_transcript", "final_transcript",
    "status", "review_status", "automatic_status",
    "segment_id", "audio_file",
    "changed_by", "changed_at", "change_log",
]
df_out = df[[c for c in APP_COLS if c in df.columns]].copy()

df_out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print(df_out["status"].value_counts().to_dict())
print("Saved (exact app format):", OUT_CSV)
print("Columns:", list(df_out.columns))
df_out[["segment_id", "model_A_transcript", "model_B_transcript", "status", "automatic_status"]].head(10)

{'difficult': 226, 'human_choice': 108, 'accepted': 52}
Saved (exact app format): /kaggle/working/asr_out/segments.csv
Columns: ['start', 'end', 'label', 'snr', 'quality', 'model_A_transcript', 'model_B_transcript', 'final_transcript', 'status', 'review_status', 'automatic_status', 'segment_id', 'audio_file', 'changed_by', 'changed_at', 'change_log']


,segment_id,model_A_transcript,model_B_transcript,status,automatic_status
0,1,عن النبي صلى الله عليه وسلم يأتي علينا الزمان ...,يعني النبي صلى الله عليه وسلم يأتي على الناس ز...,difficult,difficult
1,2,ومن لم يأكل الربا أصابه وباره. إذا تشوف بعض الناس,ومن لم يأكل ربا أصابه وباره، لذلك شوف بعض النصوص,difficult,difficult
2,3,بعض الناس تاجر وغني وما عنده مشكلة أبداً يلعب ...,بعض الناس تاجر وغني وما عنده مشكلة أبدا يلعب ي...,difficult,difficult
3,4,قبل الرجل قل ليتني مكانه من كل ما يشوف من الفت...,قبل رجلك قولي تاني مكانه من كل يوم نشوف من الف...,difficult,difficult
4,5,كل هذا ما الحرام الحمد لله وكفى وأصلي وأسلم عل...,كل هذا ما الحرام؟ الحمد لله وكفى وأصلي وأسلم ع...,accepted,accepted
5,6,يقول الله تبارك وتعالى أَحَسِبَ النَّاسُ أَنْ ...,يقول الله تبارك وتعالى: [ألف لام ميم] أحسب الن...,human_choice,human_choice
6,7,وَلَقَدْ فَتَنَّ الَّذِينَ مِنْ قَبْلِهِمْ فَل...,ولقد فتن الذين من قبلهم فليعلمن الله الذين صدق...,human_choice,human_choice
7,8,فتن تموج كما يموج البحر من الناس من يتشربها قل...,فتن تمو ج كما يموج البحر من الناس من يتشربها ق...,accepted,accepted
8,9,ومنهم من ينكرها فؤاده فيبقى راسخا على دينه ثاب...,ومنهم من ينكرها فؤاده فيبقى راسخا على دينه ثاب...,accepted,accepted
9,10,وأنواعها وأسباب الثبات عند حصول الفتن,وأنواعها وأسباب الثبات عند حصول الفتن.,accepted,accepted
